# Week 6 · Solutions

In [ ]:
import os, sys, time, math, inspect, threading, queue, multiprocessing
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from pathlib import Path
import pandas as pd

HERE = next(p / "week06-parallelism" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "week06-parallelism" / "workers.py").exists())
sys.path.insert(0, str(HERE))          # so that "import workers" works here and in child processes
import workers

DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
assert (DATA / "students.csv").exists(), "run  python data/make_data.py  from the repository root first"
PATHS = workers.make_student_files(DATA, n_files=4)     # students1.csv ... students4.csv
print("start method:", multiprocessing.get_start_method(), "| cores:", multiprocessing.cpu_count(),
      "| files:", [p.name for p in PATHS])

### Exercise 1 · How many workers?

In [ ]:
def pool_size(n_tasks):
    cores = getattr(os, "process_cpu_count", multiprocessing.cpu_count)()
    return max(1, min(n_tasks, cores))

In [ ]:
cores = getattr(os, "process_cpu_count", multiprocessing.cpu_count)()
assert pool_size(1) == 1, "one task needs one worker"
assert pool_size(0) == 1, "always start at least one worker"
assert pool_size(1000) == cores, f"never more workers than the {cores} cores"
assert pool_size(cores + 3) == cores and pool_size(max(1, cores - 1)) == max(1, cores - 1)
print("✓ exercise 1 -", cores, "cores")

### Exercise 2 · Processes and a Queue

In [ ]:
def squares_via_processes(numbers):
    q = multiprocessing.Queue()
    procs = [multiprocessing.Process(target=workers.square_to_queue, args=(n, q)) for n in numbers]
    for p in procs:
        p.start()
    pairs = [q.get() for _ in procs]
    for p in procs:
        p.join()
    return dict(pairs)

In [ ]:
r = squares_via_processes([2, 3, 7])
assert isinstance(r, dict), "return a dict {n: n * n}"
assert r == {2: 4, 3: 9, 7: 49}, f"expected {{2: 4, 3: 9, 7: 49}}, got {r}"
assert squares_via_processes([10]) == {10: 100}
print("✓ exercise 2")

### Exercise 3 · `apply_async()` instead of `apply()`

In [ ]:
def totals_async(sizes):
    with multiprocessing.Pool(processes=2) as pool:
        pending = [pool.apply_async(workers.count_total_quiet, args=(n,)) for n in sizes]
        return [r.get() for r in pending]

In [ ]:
t0 = time.perf_counter()
r = totals_async([6, 6])
elapsed = time.perf_counter() - t0
assert r == [15, 15], f"count_total_quiet(6) is 0+1+2+3+4+5 = 15; got {r}"
assert elapsed < 2.2, f"took {elapsed:.1f} s: the two tasks ran one after the other (apply?), not at the same time (that takes at least 2.4 s)"
assert totals_async([3, 4, 5]) == [3, 6, 10], "return the results in the order of sizes"
print(f"✓ exercise 3 - {elapsed:.1f} s")

### Exercise 4 · `starmap()` with a context manager

In [ ]:
def powers(bases, exponent):
    with multiprocessing.Pool(processes=2) as pool:
        return pool.starmap(workers.power, [(b, exponent) for b in bases])

In [ ]:
r = powers([1, 2, 3, 10], 3)
assert isinstance(r, list), "starmap returns a list"
assert r == [1, 8, 27, 1000], f"expected [1, 8, 27, 1000], got {r}"
assert powers([], 2) == [] and powers([5], 0) == [1]
print("✓ exercise 4")

### Exercise 5 · Errors in workers

In [ ]:
def safe_reciprocals(xs):
    out = []
    with multiprocessing.Pool(processes=2) as pool:
        pending = [pool.apply_async(workers.reciprocal, args=(x,)) for x in xs]
        for r in pending:
            try:
                out.append(r.get())
            except ZeroDivisionError:
                out.append(None)
    return out

In [ ]:
r = safe_reciprocals([4, 0, 2, 0])
assert r == [0.25, None, 0.5, None], f"expected [0.25, None, 0.5, None], got {r}"
assert safe_reciprocals([1]) == [1.0]
print("✓ exercise 5")

### Exercise 6 · Parallel data loading (the syllabus task)

In [ ]:
def load_parallel(paths):
    with multiprocessing.Pool(processes=2) as pool:
        frames = pool.map(workers.read_csv_file, paths)
    return pd.concat(frames, ignore_index=True)

In [ ]:
assert "Pool" in inspect.getsource(load_parallel), "read the files in a process pool (multiprocessing.Pool)"
r = load_parallel(PATHS)
expected = pd.concat([pd.read_csv(p) for p in PATHS], ignore_index=True)
assert isinstance(r, pd.DataFrame), "return a DataFrame"
assert r.shape == expected.shape, f"expected shape {expected.shape}, got {r.shape}"
pd.testing.assert_frame_equal(r, expected)          # same rows, same order, index 0..n-1
assert len(load_parallel(PATHS[:2])) == len(pd.read_csv(PATHS[0])) + len(pd.read_csv(PATHS[1]))
print("✓ exercise 6 -", r.shape)

### Exercise 7 · Threads for I/O-bound work

In [ ]:
def download_all(urls):
    with ThreadPoolExecutor(max_workers=max(1, len(urls))) as ex:
        return list(ex.map(workers.fake_download, urls))

In [ ]:
urls = [f"https://example.org/page{i}" for i in range(6)]
t0 = time.perf_counter()
r = download_all(urls)
elapsed = time.perf_counter() - t0
assert isinstance(r, list) and len(r) == 6, "return one result per URL, as a list"
assert [u for u, _ in r] == urls, "keep the order of urls"
assert elapsed < 1.5, f"took {elapsed:.1f} s: the downloads did not overlap (one after the other takes 1.8 s)"
print(f"✓ exercise 7 - {elapsed:.2f} s")

### Exercise 8 · A race condition and a `Lock`

In [ ]:
def deposit_all(n_threads, n_deposits):
    account = {"balance": 0}
    lock = threading.Lock()

    def deposit():
        for _ in range(n_deposits):
            with lock:
                current = account["balance"]
                time.sleep(0.0001)              # another thread may run here
                account["balance"] = current + 1

    threads = [threading.Thread(target=deposit) for _ in range(n_threads)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return account["balance"]

print(deposit_all(4, 50), "(should be 200)")

In [ ]:
results = [deposit_all(4, 50) for _ in range(3)]
assert results == [200, 200, 200], f"expected 200 every time, got {results}: protect the read-and-write with a Lock"
assert deposit_all(3, 30) == 90
print("✓ exercise 8")

### Exercise 9 · A `Queue` of files for worker threads

In [ ]:
def count_rows(paths, n_workers=2):
    tasks, results = queue.Queue(), queue.Queue()

    def worker():
        while True:
            path = tasks.get()
            if path is None:
                break
            results.put((Path(path).name, len(pd.read_csv(path))))

    threads = [threading.Thread(target=worker) for _ in range(n_workers)]
    for t in threads:
        t.start()
    for p in paths:
        tasks.put(p)
    for _ in threads:
        tasks.put(None)
    for t in threads:
        t.join()
    return dict(results.get() for _ in range(results.qsize()))

In [ ]:
before = threading.active_count()
r = count_rows(PATHS, n_workers=2)
expected = {p.name: len(pd.read_csv(p)) for p in PATHS}
assert isinstance(r, dict), "return a dict {file name: rows}"
assert r == expected, f"expected {expected}, got {r}"
assert count_rows(PATHS[:1], n_workers=3) == {PATHS[0].name: expected[PATHS[0].name]}, "more workers than files must also work"
assert threading.active_count() <= before, "every worker thread must stop (one sentinel per worker)"
print("✓ exercise 9", r)

### Exercise 10 · `ProcessPoolExecutor` for CPU-bound work

In [ ]:
def factorial_sizes(numbers):
    with ProcessPoolExecutor(max_workers=2) as ex:
        return list(ex.map(workers.factorial_digits, numbers))

In [ ]:
assert "ProcessPoolExecutor" in inspect.getsource(factorial_sizes), "use a ProcessPoolExecutor"
sys.set_int_max_str_digits(0)
nums = [10, 100, 1000, 5000]
r = factorial_sizes(nums)
assert isinstance(r, list), "return a list"
assert r == [len(str(math.factorial(n))) for n in nums], f"wrong digit counts: {r}"
assert r[:2] == [7, 158], "10! = 3628800 has 7 digits"
print("✓ exercise 10", r)